### Population England and Spain per year

In [31]:
import pandas as pd

df_en = pd.read_csv('../Extacted data/Population/population_england2020-2024.csv')
df_sp = pd.read_csv('../Extacted data/Population/Spain_population.csv')

In [32]:
df_sp

,Unnamed: 0,"January 1, 2025","October 1, 2024","July 1, 2024","April 1, 2024","January 1, 2024","October 1, 2023","July 1, 2023","April 1, 2023","January 1, 2023","October 1, 2022","July 1, 2022","April 1, 2022","January 1, 2022","October 1, 2021","July 1, 2021","April 1, 2021","January 1, 2021"
0,Total,"49,128,297","48,999,880.00","48,821,936","48,701,130","48,619,695","48,486,865","48,320,520","48,205,962","48,085,361","47,940,295","47,781,354","47,609,145","47,486,727","47,424,595","47,346,836","47,356,065","47,400,798"
1,Men,"24,090,369","24,034,961.00","23,940,312","23,873,843","23,826,871","23,764,562","23,681,931","23,624,765","23,565,593","23,498,435","23,421,925","23,343,774","23,288,747","23,263,694",23.223.390,"23,226,336","23,248,611"
2,Women,"25,037,928","24,964,919.00","24,881,624","24,827,287","24,792,824","24,722,303","24,638,589","24,581,197","24,519,768","24,441,860","24,359,429","24,265,371","24,197,980",24.160.901,24.123.446,"24,129,729",24.152.187


In [33]:
# Set index to category column and transpose
df_sp = df_sp.set_index('Unnamed: 0').T.reset_index()
df_sp.columns = ['date', 'population_total', 'men', 'women']

# Convert date to proper format
df_sp['date'] = pd.to_datetime(df_sp['date']).dt.strftime('%Y-%m')

df_sp.head(2)

,date,population_total,men,women
0,2025-01,"49,128,297","24,090,369","25,037,928"
1,2024-10,"48,999,880.00","24,034,961.00","24,964,919.00"


In [34]:
def clean_number(x):
    x = str(x).strip()
    # Count dots and commas
    if x.count('.') > 1:  # dots as thousands separator e.g. 23.223.390
        x = x.replace('.', '')
    elif ',' in x and '.' in x:  # mixed e.g. 24,034,961.00
        x = x.replace(',', '')
    elif ',' in x:  # comma as thousands e.g. 24,090,369
        x = x.replace(',', '')
    return int(float(x))

for col in ['population_total', 'men', 'women']:
    df_sp[col] = df_sp[col].apply(clean_number)

print(df_sp.head())

      date  population_total       men     women
0  2025-01          49128297  24090369  25037928
1  2024-10          48999880  24034961  24964919
2  2024-07          48821936  23940312  24881624
3  2024-04          48701130  23873843  24827287
4  2024-01          48619695  23826871  24792824


In [35]:
df_sp_yearly = df_sp[df_sp['date'].str.endswith('-01')][['date', 'population_total']].copy()
df_sp_yearly['Year'] = df_sp_yearly['date'].str[:4].astype(int)
df_sp_yearly = df_sp_yearly[['Year', 'population_total']].rename(columns={'population_total': 'Population'})
df_sp_yearly = df_sp_yearly.sort_values('Year').reset_index(drop=True)
print(df_sp_yearly)

   Year  Population
0  2021    47400798
1  2022    47486727
2  2023    48085361
3  2024    48619695
4  2025    49128297


In [36]:
df_en = df_en.drop(columns=["Yearly Change","Net Migration","Median Age","Fertility Rate"])
# add new row
df_en.loc[len(df_en)] = [2025, 59050000]
df_en = df_en.drop(0)
df_en

,Year,Population
1,2021,56489800
2,2022,57106398
3,2023,57690300
4,2024,58397300
5,2025,59050000


In [37]:
df_en['country']= 'England'
df_sp_yearly['country']= 'Spain'

In [38]:
df_2 = pd.concat([df_en, df_sp_yearly])
df_2 = df_2.rename(columns={'Year': 'year' , 'Population' : 'population'})
df_2

,year,population,country
1,2021,56489800,England
2,2022,57106398,England
3,2023,57690300,England
4,2024,58397300,England
5,2025,59050000,England
0,2021,47400798,Spain
1,2022,47486727,Spain
2,2023,48085361,Spain
3,2024,48619695,Spain
4,2025,49128297,Spain


In [39]:
df_2.to_csv("All_population_yearly.csv", index=False)

In [40]:
# Sort by date
df_sp = df_sp.sort_values('date').reset_index(drop=True)

# Create full monthly date range
monthly_dates = pd.date_range(start='2021-01', end='2025-12', freq='MS').strftime('%Y-%m')

# Set date as index and reindex to monthly
df_sp_monthly = df_sp.set_index('date').reindex(monthly_dates)

# Interpolate missing months linearly
df_sp_monthly = df_sp_monthly.interpolate(method='linear').round(0).astype(int)

df_sp_monthly = df_sp_monthly.reset_index().rename(columns={'index': 'date'})
df_sp_monthly['country'] = 'Spain'

df_sp_monthly = df_sp_monthly[['date', 'population_total', 'country']]
df_sp_monthly = df_sp_monthly.rename(columns={'population_total': 'population'})

df_sp_monthly.head(2)

,date,population,country
0,2021-01,47400798,Spain
1,2021-02,47385887,Spain


In [41]:
# Create full monthly date range
monthly_dates = pd.date_range(start='2021-01', end='2025-12', freq='MS').strftime('%Y-%m')

# Set year as index, reindex to monthly
df_en['date'] = df_en['Year'].astype(str) + '-01'
df_en_monthly = df_en.set_index('date')[['Population']].reindex(monthly_dates)

# Interpolate missing months
df_en_monthly = df_en_monthly.interpolate(method='linear').round(0).astype(int)
df_en_monthly = df_en_monthly.reset_index().rename(columns={'index': 'date', 'Population': 'population'})
df_en_monthly['country'] = 'England'

# Calculate average monthly growth from previous years
monthly_growth = (df_en_monthly['population'].diff().dropna()).mean()

# Find 2025-01 value
pop_2025_jan = df_en_monthly[df_en_monthly['date'] == '2025-01']['population'].values[0]

# Update 2025 months with growth estimate
for i, row in df_en_monthly[df_en_monthly['date'].str.startswith('2025')].iterrows():
    month_num = int(row['date'].split('-')[1]) - 1
    df_en_monthly.loc[i, 'population'] = round(pop_2025_jan + monthly_growth * month_num)

df_en_monthly.head(2)

,date,population,country
0,2021-01,56489800,England
1,2021-02,56541183,England


In [42]:
df_pop_by_month = pd.concat([df_en_monthly, df_sp_monthly])
df_pop_by_month

,date,population,country
0,2021-01,56489800,England
1,2021-02,56541183,England
2,2021-03,56592566,England
3,2021-04,56643950,England
4,2021-05,56695333,England
...,...,...,...
55,2025-08,49128297,Spain
56,2025-09,49128297,Spain
57,2025-10,49128297,Spain
58,2025-11,49128297,Spain


In [43]:
df_pop_by_month.to_csv("All_population_monthly.csv", index=False)

### Population England per city per year

In [44]:
xls_c = pd.read_excel('../Extacted data/Population/england_city_populations_long_table.xlsx', sheet_name=None)
print(xls_c.keys())

dict_keys(['Long_Table'])


In [45]:
df_c = xls_c['Long_Table']
df_c

,City,Year,Population
0,London,2021,8799800
1,London,2022,8870000
2,London,2023,8945310
3,London,2024,9020000
4,London,2025,9095000
...,...,...,...
80,Kendal,2021,28940
81,Kendal,2022,29000
82,Kendal,2023,29100
83,Kendal,2024,29200


In [46]:
import numpy as np
import pandas as pd

def expand_to_monthly_city(df_pop):

    monthly_records = []

    for city in df_pop['City'].unique():

        df_city = (
            df_pop[df_pop['City'] == city]
            .sort_values('Year')
        )

        # interpolate existing years
        for i in range(len(df_city) - 1):

            year_start = df_city.iloc[i]['Year']

            pop_start = df_city.iloc[i]['Population']
            pop_end = df_city.iloc[i + 1]['Population']

            for month in range(1, 13):

                fraction = (month - 1) / 12

                population = (
                    pop_start +
                    (pop_end - pop_start) * fraction
                )

                monthly_records.append({
                    'date': f'{year_start}-{month:02d}',
                    'city': city,
                    'population': round(population)
                })

        # ---- generate full last year ----

        populations = df_city['Population'].values

        annual_growth_rates = [
            (populations[i+1] - populations[i]) / populations[i]
            for i in range(len(populations)-1)
        ]

        avg_growth_rate = np.mean(annual_growth_rates)

        last = df_city.iloc[-1]

        last_year = last['Year']
        last_pop = last['Population']

        projected_next_pop = (
            last_pop * (1 + avg_growth_rate)
        )

        for month in range(1, 13):

            fraction = (month - 1) / 12

            population = (
                last_pop +
                (projected_next_pop - last_pop) * fraction
            )

            monthly_records.append({
                'date': f'{last_year}-{month:02d}',
                'city': city,
                'population': round(population)
            })

    return pd.DataFrame(monthly_records)


df_monthly_city = expand_to_monthly_city(df_c)

df_monthly_city.tail(5)

,date,city,population
1015,2025-08,Kendal,29410
1016,2025-09,Kendal,29419
1017,2025-10,Kendal,29428
1018,2025-11,Kendal,29436
1019,2025-12,Kendal,29445


In [47]:
df_monthly_city.to_csv("England_population_per_city.csv", index=False)